In [2]:
# Install missing dependencies in Colab runtime before running the pipeline cells
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("segment_anything", "segment-anything"),
    ("cv2", "opencv-python"),
    ("diffusers", "diffusers"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
    ("pyngrok", "pyngrok"),
    ("nest_asyncio", "nest_asyncio"),
]

missing_pkgs = [pip_name for mod_name, pip_name in REQUIRED if importlib.util.find_spec(mod_name) is None]
if missing_pkgs:
    print("Installing missing packages:", ", ".join(missing_pkgs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])
else:
    print("All required packages already installed.")

# Some runtimes may still miss segment-anything from PyPI mirror; use GitHub fallback.
if importlib.util.find_spec("segment_anything") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/facebookresearch/segment-anything.git",
    ])
    print("Installed segment-anything from GitHub fallback.")
else:
    print("segment_anything import is available.")

All required packages already installed.
segment_anything import is available.


In [ ]:
import os, gc, json, torch, io, base64
import numpy as np
import cv2
import urllib.request
from PIL import Image
from diffusers import (
    StableDiffusionPipeline, StableDiffusionControlNetPipeline,
    ControlNetModel, StableDiffusionXLControlNetInpaintPipeline,
)
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
from transformers import CLIPModel, CLIPProcessor

# --- CONFIG ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

if "MODEL_CACHE" not in globals():
    MODEL_CACHE = {}

def load_all_models():
    print("Pre-loading models for stability...")
    # 1. SD 1.5
    if "sd15" not in globals():
        print("- Loading SD 1.5 Base...")
        global sd15_pipe
        sd15_pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=DTYPE, safety_checker=None, low_cpu_mem_usage=True).to(DEVICE)
        sd15_pipe.enable_attention_slicing()
    # 2. ControlNet
    if "cn_pipe" not in globals():
        print("- Loading ControlNet...")
        m = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=DTYPE).to(DEVICE)
        global cn_pipe
        cn_pipe = StableDiffusionControlNetPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", controlnet=m, torch_dtype=DTYPE).to(DEVICE)
        cn_pipe.enable_model_cpu_offload()
    # 3. SAM
    if "sam_gen" not in globals():
        print("- Loading SAM...")
        ckpt = "sam_vit_b.pth"
        if not os.path.exists(ckpt): urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", ckpt)
        global sam_gen
        s = sam_model_registry["vit_b"](checkpoint=ckpt).to(DEVICE)
        sam_gen = SamAutomaticMaskGenerator(s, points_per_side=8)

def _json_safe(obj):
    if isinstance(obj, dict): return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list): return [_json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, (np.float16, np.float32, np.float64)): return float(obj)
    return obj

def run_pipeline_once(payload):
    p = payload.get("prompt", "interior")
    np_p = payload.get("negative_prompt", "")
    print("--- Stage 1: Base Image ---")
    img_base = sd15_pipe(prompt=p, negative_prompt=np_p, num_inference_steps=12).images[0]
    print("--- Stage 2: Segmentation ---")
    masks = sorted(sam_gen.generate(np.array(img_base)), key=lambda x: x["area"], reverse=True)[:3]
    results = [{"label": f"object_{i}", "confidence": 0.9, "bbox": [float(v) for v in m["bbox"]]} for i, m in enumerate(masks)]
    print("--- Stage 3: ControlNet Edit ---")
    gray = np.array(img_base.convert("L")); edges = cv2.Canny(gray, 100, 200); canny = Image.fromarray(edges).convert("RGB")
    img_edit = cn_pipe(prompt=p, negative_prompt=np_p, image=canny, num_inference_steps=15).images[0]
    print("--- Pipeline Complete ---")
    return img_base, _json_safe(results), img_edit, img_edit, False

load_all_models()


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Safe Pipeline Ready.


In [4]:
# Force SD 1.5 family to load fp16 weights on CUDA.
# Run this after Cell 2. It overrides loaders and clears stale SD1.5 cache entries.

if DEVICE != "cuda":
    print("CUDA not available; fp16 SD1.5 loading is skipped.")
else:
    def _from_pretrained_fp16(loader, model_id, **kwargs):
        # Prefer explicit fp16 variant; fall back to fp16 dtype if variant is unavailable.
        try:
            return loader(
                model_id,
                torch_dtype=torch.float16,
                variant="fp16",
                use_safetensors=True,
                **kwargs,
            )
        except Exception:
            return loader(
                model_id,
                torch_dtype=torch.float16,
                **kwargs,
            )

    def _get_sd15_pipe():
        key = _cache_key("sd15")
        if key not in MODEL_CACHE:
            _log_cache("sd15", key, loaded=True)
            pipe = _from_pretrained_fp16(
                StableDiffusionPipeline.from_pretrained,
                "runwayml/stable-diffusion-v1-5",
                safety_checker=None,
            )
            MODEL_CACHE[key] = _optimize_pipe(pipe)
        else:
            _log_cache("sd15", key, loaded=False)
        return MODEL_CACHE[key]

    def _get_cn_pipe():
        key = _cache_key("sd15_controlnet")
        if key not in MODEL_CACHE:
            _log_cache("sd15_controlnet", key, loaded=True)
            cn_model = _from_pretrained_fp16(
                ControlNetModel.from_pretrained,
                "lllyasviel/sd-controlnet-canny",
            ).to(DEVICE)
            cn_pipe = _from_pretrained_fp16(
                StableDiffusionControlNetPipeline.from_pretrained,
                "runwayml/stable-diffusion-v1-5",
                controlnet=cn_model,
            )
            MODEL_CACHE[key] = _optimize_pipe(cn_pipe)
        else:
            _log_cache("sd15_controlnet", key, loaded=False)
        return MODEL_CACHE[key]

    # Remove previously cached SD1.5 artifacts so next request reloads in fp16.
    removed = 0
    for k in list(MODEL_CACHE.keys()):
        if k.startswith("sd15:") or k.startswith("sd15_controlnet:"):
            MODEL_CACHE.pop(k, None)
            removed += 1

    print(f"SD1.5 fp16 override active. Cleared {removed} SD1.5 cache entries.")

SD1.5 fp16 override active. Cleared 0 SD1.5 cache entries.


In [5]:
# Required for DNS-safe tunnel from many local networks
# Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
os.environ["NGROK_AUTHTOKEN"] = "3BNZL8TUwc4xxXu11SZo0FAOwNL_52Zr8YDairH7xkWcD5NqD"  # e.g. 2abc...
os.environ["ALLOW_CLOUDFLARE_FALLBACK"] = "false"  # keep false on networks that cannot resolve trycloudflare.com

if not os.environ["NGROK_AUTHTOKEN"].strip():
    raise RuntimeError(
        "NGROK_AUTHTOKEN is empty. Paste your token in this cell, then run again. "
        "Do not proceed to the bridge cell until this passes."
    )

masked = os.environ["NGROK_AUTHTOKEN"][:6] + "..."
print("NGROK_AUTHTOKEN loaded:", masked)
print("Now run the next cell to start bridge + ngrok tunnel.")

NGROK_AUTHTOKEN loaded: 3BNZL8...
Now run the next cell to start bridge + ngrok tunnel.


In [ ]:
# Colab API server for backend integration
!pip -q install fastapi uvicorn pyngrok nest_asyncio

import io
import os
import re
import time
import json
import base64
import socket
import threading
import subprocess
from typing import Any, Dict

import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from pyngrok import ngrok

nest_asyncio.apply()


def _sanitize_steps(value, default, min_value=5, max_value=100):
    try:
        parsed = int(value)
    except Exception:
        return default
    return max(min_value, min(max_value, parsed))


class GeneratePayload(BaseModel):
    prompt: str = "a forest at sunset"
    negative_prompt: str = ""
    inpaint_prompt: str | None = None
    mask_b64: str | None = None
    enable_inpaint: bool = False
    room_type: str = "generic"
    steps_sd15: int = Field(default=12, ge=5, le=100)
    steps_sdxl: int = Field(default=10, ge=5, le=100)
    job_id: str | None = None
    low_vram: bool = True
    cache_debug: bool = True


app = FastAPI(title="AI Image Studio Colab Bridge")


def _pipeline_loaded() -> bool:
    return callable(globals().get("run_pipeline_once"))


def _current_device() -> str:
    return str(globals().get("DEVICE", "unknown"))


def _img_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": _current_device(),
        "pipeline_loaded": _pipeline_loaded(),
    }


@app.post("/generate")
def generate_api(req: GeneratePayload):
    if not _pipeline_loaded():
        raise HTTPException(
            status_code=503,
            detail=(
                "Pipeline functions are not loaded in this runtime. "
                "Run Cell 2 (pipeline definitions), then re-run Cell 5 (bridge server)."
            ),
        )

    payload: Dict[str, Any] = req.model_dump()
    payload["inpaint_prompt"] = payload.get("inpaint_prompt") or payload["prompt"]
    payload.setdefault("low_vram", True)
    payload.setdefault("enable_inpaint", False)
    payload.setdefault("room_type", "generic")
    payload["steps_sd15"] = _sanitize_steps(payload.get("steps_sd15", 12), 12)
    payload["steps_sdxl"] = _sanitize_steps(payload.get("steps_sdxl", 10), 10)
    payload.setdefault("cache_debug", True)
    os.environ["PIPELINE_PAYLOAD"] = json.dumps(payload)

    try:
        runner = globals()["run_pipeline_once"]
        img_base, labels, img_edited, img_final, inpaint_applied = runner(payload)

        return {
            "job_id": payload.get("job_id"),
            "base_image": _img_to_b64(img_base),
            "edited_image": _img_to_b64(img_edited),
            "final_image": _img_to_b64(img_final),
            "inpaint_applied": inpaint_applied,
            "segments": labels,
            "execution_mode": "colab",
            "gpu_device": _current_device(),
            "low_vram": payload.get("low_vram", True),
            "enable_inpaint": payload.get("enable_inpaint", False),
            "room_type": payload.get("room_type", "generic"),
            "steps_sd15": payload.get("steps_sd15", 12),
            "steps_sdxl": payload.get("steps_sdxl", 10),
            "cache_debug": payload.get("cache_debug", True),
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))


def _ensure_cloudflared_binary():
    if subprocess.run(["bash", "-lc", "command -v cloudflared >/dev/null 2>&1"], check=False).returncode == 0:
        return
    subprocess.run(
        [
            "bash",
            "-lc",
            "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        ],
        check=True,
    )


def _open_cloudflare_tunnel(port: int):
    _ensure_cloudflared_binary()
    log_path = "/tmp/cloudflared.log"
    try:
        os.remove(log_path)
    except FileNotFoundError:
        pass

    subprocess.run(["bash", "-lc", "pkill -f 'cloudflared tunnel --url'"], check=False)

    proc = subprocess.Popen(
        [
            "cloudflared",
            "tunnel",
            "--url",
            f"http://127.0.0.1:{port}",
            "--no-autoupdate",
            "--logfile",
            log_path,
            "--loglevel",
            "info",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
        text=True,
    )

    pattern = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")
    for _ in range(120):
        time.sleep(1)
        if os.path.exists(log_path):
            content = open(log_path, "r", encoding="utf-8", errors="ignore").read()
            match = pattern.search(content)
            if match:
                return match.group(0).rstrip("/"), proc
        if proc.poll() is not None:
            break

    tail = ""
    if os.path.exists(log_path):
        lines = open(log_path, "r", encoding="utf-8", errors="ignore").read().splitlines()
        tail = "\n".join(lines[-20:])
    proc.terminate()
    raise RuntimeError("Failed to obtain Cloudflare tunnel URL. Cloudflared logs:\n" + (tail or "<no logs>"))


def _open_public_tunnel(port: int):
    # Prefer ngrok because some local networks cannot resolve trycloudflare.com domains.
    ngrok_token = os.getenv("NGROK_AUTHTOKEN") or os.getenv("COLAB_NGROK_AUTHTOKEN")
    allow_cloudflare_fallback = os.getenv("ALLOW_CLOUDFLARE_FALLBACK", "false").lower() == "true"

    if not ngrok_token and not allow_cloudflare_fallback:
        raise RuntimeError(
            "NGROK_AUTHTOKEN is not set. This notebook now prefers ngrok to avoid DNS failures on trycloudflare domains. "
            "Set NGROK_AUTHTOKEN in Colab, then re-run this cell. "
            "If you still want Cloudflare fallback, set ALLOW_CLOUDFLARE_FALLBACK=true."
        )

    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)

        # Clear any existing tunnels/processes that may hold a named endpoint.
        try:
            for t in ngrok.get_tunnels():
                ngrok.disconnect(t.public_url)
        except Exception:
            pass
        try:
            ngrok.kill()
        except Exception:
            pass
        subprocess.run(["bash", "-lc", "pkill -f ngrok"], check=False)
        try:
            url = ngrok.connect(addr=str(port), proto="http").public_url.rstrip("/")
            return url, "ngrok", None
        except Exception as ngrok_exc:
            print("ngrok failed:", str(ngrok_exc))
            msg = str(ngrok_exc)
            if "ERR_NGROK_334" in msg or "already online" in msg:
                print("Detected existing ngrok endpoint. Retrying with pooling enabled...")
                try:
                    ngrok.kill()
                    time.sleep(2)
                    pooled_url = ngrok.connect(
                        addr=str(port),
                        proto="http",
                        pooling_enabled=True,
                        bind_tls=True,
                    ).public_url.rstrip("/")
                    return pooled_url, "ngrok", None
                except Exception as pooled_exc:
                    print("ngrok pooled retry failed:", str(pooled_exc))
                    try:
                        print("Retrying ngrok with unique tunnel name...")
                        ngrok.kill()
                        time.sleep(2)
                        named_url = ngrok.connect(
                            addr=str(port),
                            proto="http",
                            name=f"colab-bridge-{int(time.time())}",
                            bind_tls=True,
                        ).public_url.rstrip("/")
                        return named_url, "ngrok", None
                    except Exception as named_exc:
                        print("ngrok named retry failed:", str(named_exc))
            if not allow_cloudflare_fallback:
                raise RuntimeError("ngrok failed and Cloudflare fallback is disabled.") from ngrok_exc

    print("Using Cloudflare quick tunnel fallback.")
    cf_url, cf_proc = _open_cloudflare_tunnel(port)
    return cf_url, "cloudflared", cf_proc


def _is_port_available(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        return s.connect_ex(("127.0.0.1", port)) != 0


def _pick_available_port(start_port: int, max_tries: int = 25) -> int:
    for offset in range(max_tries):
        candidate = start_port + offset
        if _is_port_available(candidate):
            return candidate
    raise RuntimeError(f"No free port found starting from {start_port}.")


requested_port = int(os.environ.get("COLAB_BRIDGE_PORT", "8001"))
port = _pick_available_port(requested_port)
if port != requested_port:
    print(f"Requested port {requested_port} is in use; using free port {port} instead.")
active_port = globals().get("server_port")
thread_running = "thread" in globals() and getattr(thread, "is_alive", lambda: False)()

if thread_running and active_port == port:
    print(f"API server already running on port {port}.")
else:
    if thread_running and active_port != port:
        print(f"Detected stale server on port {active_port}; starting a new server on port {port}.")
    thread = threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=port, log_level="info"),
        daemon=True,
        name="colab-fastapi-server",
    )
    thread.start()
    server_port = port

public_base_url, tunnel_provider, cloudflared_proc = _open_public_tunnel(port)
print("Tunnel provider:", tunnel_provider)
print("Public base URL:", public_base_url)
print("Set COLAB_SERVER_URL=", f"{public_base_url}/generate")
print("Set COLAB_HEALTHCHECK_URL=", f"{public_base_url}/health")
if tunnel_provider == "ngrok":
    print("ngrok URL ready. Use these values in your local .env.")

Requested port 8001 is in use; using free port 8002 instead.
Detected stale server on port 8001; starting a new server on port 8002.


INFO:     Started server process [23571]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8002 (Press CTRL+C to quit)


Tunnel provider: ngrok
Public base URL: https://spongiest-noneagerly-ursula.ngrok-free.dev
Set COLAB_SERVER_URL= https://spongiest-noneagerly-ursula.ngrok-free.dev/generate
Set COLAB_HEALTHCHECK_URL= https://spongiest-noneagerly-ursula.ngrok-free.dev/health
ngrok URL ready. Use these values in your local .env.


INFO:     59.184.213.244:0 - "GET /health HTTP/1.1" 200 OK
--- 1. Generating Base ---
INFO:     59.184.213.244:0 - "POST /generate HTTP/1.1" 500 Internal Server Error
INFO:     59.184.213.244:0 - "GET /health HTTP/1.1" 200 OK
--- 1. Generating Base ---
INFO:     59.184.213.244:0 - "POST /generate HTTP/1.1" 500 Internal Server Error
